# Variables and data types in python 

## Libraries and settings

In [10]:
# Libraries
import os
import random
import numpy as np
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

/Users/drinmuslija/IdeaProjects/scientific_programming/Week_02/exercises


## Creating an Excel file containing simulated chicken data
(yes, this is not neccessary because we have Python, but it shows how we can work with Excel spread sheets from within Python)

In [11]:
# Empty list to store the data
col_01 = []
col_02 = []
col_03 = []
col_04 = []

# List with chicken breeds
breed = ['ISA Brown',
         'Plymouth Rock',
         'Barnevelder',
         'Australorp',
         'New Hampshire Red']

# Fill the empty lists with data
random.seed(10)
for i in range(1, 101):
    col_01.append(i)
    col_02.append(float(np.random.normal(2500, 250, 1)))
    col_03.append(breed[random.randint(0, 4)])
    col_04.append(random.randint(150, 365))

# Create a data frame from the lists
df = pd.DataFrame.from_dict({'chicken_id': col_01,
                             'weight': col_02,
                             'breed': col_03,
                             'eggs_per_year': col_04})

################################################################################

# Create a Pandas Excel writer using XlsxWriter as the engine
writer = pd.ExcelWriter('./data/chicken_data.xlsx', engine='xlsxwriter')

# Convert the dataframe to an XlsxWriter Excel object
df.to_excel(writer, sheet_name='Sheet1', startrow=1, header=False, index=False)

# Get the xlsxwriter objects from the dataframe writer object
workbook = writer.book
worksheet = writer.sheets['Sheet1']

# Get the dimensions of the dataframe
(max_row, max_col) = df.shape

# Create a list of column headers, to use in add_table()
column_settings = [{'header': column} for column in df.columns]

# Add the Excel table structure. Pandas will add the data
worksheet.add_table(0, 0, max_row, max_col - 1, {'columns': column_settings})

# Make the columns wider for clarity
worksheet.set_column(0, max_col - 1, 20)

################################################################################

# Create pivot table
table = pd.pivot_table(df[['breed', 'eggs_per_year']],
                       index=['breed'],
                       values=['eggs_per_year'],
                       aggfunc=np.mean)

table.to_excel(writer,
               sheet_name='Sheet2',
               startrow=0,
               header=True,
               index=True)

# Include barchart
chart = workbook.add_chart({'type': 'bar', 'subtype': 'stacked'})

# Configure the first series.
chart.add_series({
    'name':       '=Sheet2!$B$1',
    'categories': '=Sheet2!$A$2:$A$6',
    'values':     '=Sheet2!$B$2:$B$6',
})

# Add a chart title and some axis labels.
chart.set_title({'name': 'Average eggs per breed'})
chart.set_x_axis({'name': ''})
chart.set_y_axis({'name': ''})

# Set an Excel chart style.
chart.set_style(13)

# Insert the chart into the worksheet (with an offset).
worksheet.insert_chart('F1', chart, {'x_offset': 25, 'y_offset': 10})

# Insert .png - file
worksheet.insert_image('H17', './data/chickens.jpg')

# Close the Pandas Excel writer and output the Excel file.
writer.close()

## Read data from Excel-File

In [14]:
# Read xlsx file
df = pd.read_excel('./data/chicken_data.xlsx', sheet_name='Sheet1')
df

,chicken_id,weight,breed,eggs_per_year
0,1,2336.190843,New Hampshire Red,158
1,2,2510.979184,Australorp,273
2,3,2714.543305,New Hampshire Red,153
3,4,2494.320175,Plymouth Rock,268
4,5,2105.180791,Australorp,360
...,...,...,...,...
95,96,2267.717020,Plymouth Rock,314
96,97,2225.680779,New Hampshire Red,297
97,98,2633.972865,Australorp,156
98,99,2050.917958,ISA Brown,225


## Explore data types

In [15]:
# Explore data types (note that 'object' means categorical in pandas)
df.dtypes

chicken_id         int64
weight           float64
breed             object
eggs_per_year      int64
dtype: object

## Change data type of variable 'weight'

In [16]:
# Change data type (note that with .astype('int32'), the values are simply cutted!)
df['weight_new'] =  df['weight'].astype('int32')
print(df.dtypes, '\n')

df

chicken_id         int64
weight           float64
breed             object
eggs_per_year      int64
weight_new         int32
dtype: object 



,chicken_id,weight,breed,eggs_per_year,weight_new
0,1,2336.190843,New Hampshire Red,158,2336
1,2,2510.979184,Australorp,273,2510
2,3,2714.543305,New Hampshire Red,153,2714
3,4,2494.320175,Plymouth Rock,268,2494
4,5,2105.180791,Australorp,360,2105
...,...,...,...,...,...
95,96,2267.717020,Plymouth Rock,314,2267
96,97,2225.680779,New Hampshire Red,297,2225
97,98,2633.972865,Australorp,156,2633
98,99,2050.917958,ISA Brown,225,2050


## Create new variable 'breed_str' which has string as the data type

In [17]:
df['breed_str'] = pd.Series(['breed'], dtype="string")
df.dtypes

chicken_id                int64
weight                  float64
breed                    object
eggs_per_year             int64
weight_new                int32
breed_str        string[python]
dtype: object

## Create new categorical variable from 'eggs_per_year'

In [18]:
# Define labels for categories
labels = ['0 - 99', '100 - 199', '20 - 365']

# Create categories
df["eggs_cat"] = pd.cut(df['eggs_per_year'], bins=[0, 100, 200, 365], labels=labels)
df[['eggs_per_year', 'eggs_cat']].head(10)

,eggs_per_year,eggs_cat
0,158,100 - 199
1,273,20 - 365
2,153,100 - 199
3,268,20 - 365
4,360,20 - 365
5,317,20 - 365
6,158,100 - 199
7,275,20 - 365
8,169,100 - 199
9,340,20 - 365


## Create new numerical variable from 'weight'

In [19]:
# Create new variable
df['weight_kg'] = round(df['weight'] / 1000, 4)

# Show values
df

,chicken_id,weight,breed,eggs_per_year,weight_new,breed_str,eggs_cat,weight_kg
0,1,2336.190843,New Hampshire Red,158,2336,breed,100 - 199,2.3362
1,2,2510.979184,Australorp,273,2510,<NA>,20 - 365,2.5110
2,3,2714.543305,New Hampshire Red,153,2714,<NA>,100 - 199,2.7145
3,4,2494.320175,Plymouth Rock,268,2494,<NA>,20 - 365,2.4943
4,5,2105.180791,Australorp,360,2105,<NA>,20 - 365,2.1052
...,...,...,...,...,...,...,...,...
95,96,2267.717020,Plymouth Rock,314,2267,<NA>,20 - 365,2.2677
96,97,2225.680779,New Hampshire Red,297,2225,<NA>,20 - 365,2.2257
97,98,2633.972865,Australorp,156,2633,<NA>,100 - 199,2.6340
98,99,2050.917958,ISA Brown,225,2050,<NA>,20 - 365,2.0509


## Transform categorical variable 'breed' to matrix with binary (0/1) values

In [20]:
# Use the get_dummies() method from pandas for conversion
df_02 = pd.get_dummies(df, drop_first=False, columns=['breed'], dtype='int')
df_02


,chicken_id,weight,eggs_per_year,weight_new,breed_str,eggs_cat,weight_kg,breed_Australorp,breed_Barnevelder,breed_ISA Brown,breed_New Hampshire Red,breed_Plymouth Rock
0,1,2336.190843,158,2336,breed,100 - 199,2.3362,0,0,0,1,0
1,2,2510.979184,273,2510,<NA>,20 - 365,2.5110,1,0,0,0,0
2,3,2714.543305,153,2714,<NA>,100 - 199,2.7145,0,0,0,1,0
3,4,2494.320175,268,2494,<NA>,20 - 365,2.4943,0,0,0,0,1
4,5,2105.180791,360,2105,<NA>,20 - 365,2.1052,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,2267.717020,314,2267,<NA>,20 - 365,2.2677,0,0,0,0,1
96,97,2225.680779,297,2225,<NA>,20 - 365,2.2257,0,0,0,1,0
97,98,2633.972865,156,2633,<NA>,100 - 199,2.6340,1,0,0,0,0
98,99,2050.917958,225,2050,<NA>,20 - 365,2.0509,0,0,1,0,0


## Show data types of all generated variables

In [21]:
# Show data types
df_02.dtypes

chicken_id                          int64
weight                            float64
eggs_per_year                       int64
weight_new                          int32
breed_str                  string[python]
eggs_cat                         category
weight_kg                         float64
breed_Australorp                    int64
breed_Barnevelder                   int64
breed_ISA Brown                     int64
breed_New Hampshire Red             int64
breed_Plymouth Rock                 int64
dtype: object

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [22]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Darwin | 25.2.0
Datetime: 2026-03-05 19:09:57
Python Version: 3.9.6
-----------------------------------


In [26]:
now = datetime.now()
yearFrom = now.year - 100
#Show all leap years from yearFrom to current year with the day of the week of February 29th
leap_years = []
for year in range(yearFrom, now.year + 1):
    if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
        leap_years.append((year, datetime(year, 2, 29).strftime('%A')))
print('Leap years from', yearFrom, 'to', now.year, ':')
for year, day in leap_years:
    print(f'{year} (February 29th was on a {day})')

Leap years from 1926 to 2026 :
1928 (February 29th was on a Wednesday)
1932 (February 29th was on a Monday)
1936 (February 29th was on a Saturday)
1940 (February 29th was on a Thursday)
1944 (February 29th was on a Tuesday)
1948 (February 29th was on a Sunday)
1952 (February 29th was on a Friday)
1956 (February 29th was on a Wednesday)
1960 (February 29th was on a Monday)
1964 (February 29th was on a Saturday)
1968 (February 29th was on a Thursday)
1972 (February 29th was on a Tuesday)
1976 (February 29th was on a Sunday)
1980 (February 29th was on a Friday)
1984 (February 29th was on a Wednesday)
1988 (February 29th was on a Monday)
1992 (February 29th was on a Saturday)
1996 (February 29th was on a Thursday)
2000 (February 29th was on a Tuesday)
2004 (February 29th was on a Sunday)
2008 (February 29th was on a Friday)
2012 (February 29th was on a Wednesday)
2016 (February 29th was on a Monday)
2020 (February 29th was on a Saturday)
2024 (February 29th was on a Thursday)
